In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import glob
import numpy as np
import matplotlib.pyplot as plt
import optuna
import copy
import random

class CNN2D(nn.Module):
    def _get_conv_output(self, shape):
        with torch.no_grad():
            x = torch.zeros(1, *shape)
            x = self.pool1(self.conv1(x))
            x = self.pool2(self.conv2(x))
            x = self.pool3(self.conv3(x))
            return x.numel()

    def __init__(self, input_channels, dropout_rate=0.6, fc_hidden=128):
        super(CNN2D, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.pool3 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        conv_output_size = self._get_conv_output((input_channels, 200, 50))

        self.fc1 = nn.Linear(conv_output_size, fc_hidden)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(fc_hidden, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Training method for PyTorch
def train_model(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)
        loss = criterion(outputs, labels.squeeze(1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.sigmoid(outputs) >= 0.5
        correct += (preds == labels.squeeze(1).bool()).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = correct / total
    return epoch_loss, accuracy

# Validation function
def validate_model(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, labels.squeeze(1))
            running_loss += loss.item() * inputs.size(0)
            
            # Compute accuracy
            preds = torch.sigmoid(outputs) >= 0.5
            correct += (preds == labels.squeeze(1).bool()).sum().item()
            total += labels.size(0)
    
    validation_loss = running_loss / total
    accuracy = correct / total
    return validation_loss, accuracy

In [3]:
class GridDataset(Dataset):
    def __init__(self, data_dict):
        self.data = list(data_dict.values())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        grid = sample['grid_tensor']
        label = torch.tensor(sample['label'], dtype=torch.float32)
        return grid, label.unsqueeze(0)

In [4]:
# Create a dictionary with file names as keys and label + tensor grid as values
positive_grids = glob.glob('/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/PositiveWithoutSpies/*.npy')
validation_grids = glob.glob('/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/Validation_Set/*.npy')
file_data = {} # format is filename as key, label and grid tensor are values

for file in positive_grids:
    # Load the numpy array and convert it to a PyTorch tensor
    grid = np.load(file)
    grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)  # Adds channel dimension
    file_data[file] = {'label': 1, 'grid_tensor': grid_tensor}
positive_grids = file_data
print(len(positive_grids), "is length of positive")

file_data = {} # format is filename as key, label and grid tensor are values

positive_validation_count = 0
unlabeled_validation_count = 0

import random

positive_files = []
unlabeled_files = []

# Step 1: Separate files
for file in validation_grids:
    if any(f"-p{i}" in file for i in range(1, 999)):
        unlabeled_files.append(file)
    else:
        positive_files.append(file)

print("Before balancing:")
print("Positives:", len(positive_files))
print("Unlabeled:", len(unlabeled_files))

# Step 2: Balance counts
min_count = min(len(positive_files), len(unlabeled_files))

positive_files = random.sample(positive_files, min_count)
unlabeled_files = random.sample(unlabeled_files, min_count)

balanced_files = positive_files + unlabeled_files
random.shuffle(balanced_files)

# Step 3: Process balanced set
file_data = {}
positive_validation_count = 0
unlabeled_validation_count = 0

for file in balanced_files:
    grid = np.load(file)
    grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)

    if file in unlabeled_files:
        label = 0
        unlabeled_validation_count += 1
    else:
        label = 1
        positive_validation_count += 1

    file_data[file] = {'label': label, 'grid_tensor': grid_tensor}

print("After balancing:")
print("Positives:", positive_validation_count)
print("Unlabeled:", unlabeled_validation_count)

validation_grids = file_data
print(len(validation_grids), "is length of validation grids")


file_data = {} # format is filename as key, label and grid tensor are values

k = 50
subset_grids = []
for i in range(1, k + 1):
    file_data = {}
    subset_grid = glob.glob(f'/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/k_subsets/subset_{i}/*.npy')  # Adjust path as needed
    for file in subset_grid:
        # Load the numpy array and convert it to a PyTorch tensor
        grid = np.load(file)
        grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)  # Adds channel dimension
        file_data[file] = {'label': 0, 'grid_tensor': grid_tensor} # 0 means unlabeled
    subset_grid = file_data
    subset_grids.append(subset_grid)
    print(len(subset_grid), "is length of subset grid")

bins = []
for subset_grid in subset_grids:
    bin = {**positive_grids, **subset_grid} # merged
    bins.append(bin)


385 is length of positive
Before balancing:
Positives: 77
Unlabeled: 5659
After balancing:
Positives: 77
Unlabeled: 77
154 is length of validation grids
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is

In [5]:
def plot_graphs(train_losses, validation_losses, validation_accuracies, learning_rates):
    # Plot Training Loss vs Validation Loss
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
    plt.plot(range(1, len(validation_losses) + 1), validation_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss vs Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

    # Plot Validation Accuracy
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(validation_accuracies) + 1), validation_accuracies, label='Validation Accuracy', color='green')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy over Epochs')
    plt.legend()
    plt.grid()
    plt.show()

    # Plot Learning Rate
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(learning_rates) + 1), learning_rates, label='Learning Rates', color='green')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy over Epochs')
    plt.legend()
    plt.grid()
    plt.show()

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    # Reproducibility
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Hyperparameters to tune
    lr = trial.suggest_float("lr", 1e-6, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32])
    fc_hidden = trial.suggest_categorical("fc_hidden", [64, 128, 256])

    # Tuning settings
    tune_epochs = 150
    num_bins_to_use = 2

    # Randomly choose a few bins for faster tuning
    selected_bin_indices = list(range(min(num_bins_to_use, len(bins))))

    criterion = nn.BCEWithLogitsLoss()
    bin_val_losses = []

    validation_dataset = GridDataset(validation_grids)
    validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

    for bin_idx in selected_bin_indices:
        model = CNN2D(
            input_channels=1,
            dropout_rate=dropout_rate,
            fc_hidden=fc_hidden
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        train_dataset = GridDataset(bins[bin_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        best_val_loss = float("inf")

        for epoch in range(tune_epochs):
            train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = validate_model(model, validation_loader, criterion, device)

            # Report intermediate value for pruning
            trial.report(val_loss, step=bin_idx * tune_epochs + epoch)

            if trial.should_prune():
                raise optuna.TrialPruned()

            # if val_loss < best_val_loss:
            #     best_val_loss = val_loss

        bin_val_losses.append(val_loss)

    # Objective: minimize average best validation loss
    return float(np.mean(bin_val_losses))

sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner
)

study.optimize(objective, n_trials=30)

print("Best trial:")
print("  Value:", study.best_trial.value)
print("  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

best_params = study.best_trial.params
print(best_params)

[I 2026-04-15 11:02:33,683] A new study created in memory with name: no-name-be7197c2-1dbb-44b4-8b27-60d4ef700d9e
[W415 11:02:33.865702223 NNPACK.cpp:61] Could not initialize NNPACK! Reason: Unsupported hardware.
[I 2026-04-15 11:09:20,324] Trial 0 finished with value: 0.8073062741503046 and parameters: {'lr': 1.329291894316217e-05, 'weight_decay': 0.006351221010640704, 'dropout_rate': 0.5659969709057024, 'batch_size': 4, 'fc_hidden': 64}. Best is trial 0 with value: 0.8073062741503046.
[I 2026-04-15 11:13:13,708] Trial 1 finished with value: 0.47938626791749683 and parameters: {'lr': 1.1527987128232402e-06, 'weight_decay': 0.00757947995334801, 'dropout_rate': 0.6162213204002108, 'batch_size': 32, 'fc_hidden': 64}. Best is trial 1 with value: 0.47938626791749683.
[I 2026-04-15 11:16:58,598] Trial 2 finished with value: 1.7524953158451364 and parameters: {'lr': 6.847920095574779e-05, 'weight_decay': 3.6138942712165278e-06, 'dropout_rate': 0.34607232426760903, 'batch_size': 16, 'fc_hidde

KeyboardInterrupt: 

In [ ]:
# import os

# # Set device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Initialize the model
# criterion = nn.BCEWithLogitsLoss()

# # Define paths for saving models
# save_dir = "RDKIT-GNN-5A_Exp1"
# os.makedirs(save_dir, exist_ok=True)

# # Training loop
# epochs = 2000
# batch_size = 8

# # keep 10 positives and 10 negatives for validation data
# validation_dataset = GridDataset(validation_grids)
# validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

# for i in range(0, len(bins)):
#     model = CNN2D(input_channels=1).to(device)
#     optimizer = optim.Adam(model.parameters(), lr=0.000001, weight_decay=1e-3) 
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5000, eta_min=1e-10)

#     print(f"Training on bin {i+1}/{len(bins)}")
#     dataset = GridDataset(bins[i])
#     dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

#     train_losses = []
#     learning_rates = []
#     validation_losses = []
#     validation_accuracies = []

#     best_val_loss = float("inf")
#     best_model_state = None

#     for epoch in range(epochs):
#         epoch_loss, accuracy = train_model(model, dataloader, criterion, optimizer, device)
#         validation_loss, validation_accuracy = validate_model(model, validation_dataloader, criterion, device)
#         current_lr = optimizer.param_groups[0]['lr'] 
#         train_losses.append(epoch_loss)
#         learning_rates.append(current_lr)
#         validation_losses.append(validation_loss)
#         validation_accuracies.append(validation_accuracy)   
#         if epoch % 10 == 0:
#             print(
#                 f"Bin {i+1}, Epoch {epoch+1}/{epochs}, "
#                 f"Train Loss: {epoch_loss:.4f}, Validation Loss: {validation_loss:.4f},  "
#                 f"Accuracy: {validation_accuracy:.4f}, "
#                 f"LR: {current_lr:.6f}"
#             )

#         if validation_loss < best_val_loss:
#             best_val_loss = validation_loss
#             best_model_state = model.state_dict()
#         scheduler.step()

#     plot_graphs(train_losses, validation_losses, validation_accuracies, learning_rates)
    
#     #Save the trained model
#     model_path = os.path.join(save_dir, f"model_bin_{i+1}.pth")
#     torch.save(model.state_dict(), model_path)
#     print(f"Model for bin {i+1} saved to {model_path}")


# print("Training complete.")

